# Scheme Performance & Benchmark Analysis
**Bluestock Fintech Capstone Project**
Author: Akash Kumar Pandit


## 1. Environment Setup & Data Loading


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

sns.set_theme(style="whitegrid")
conn = sqlite3.connect('../data/processed/database.db')

scheme_perf = pd.read_sql("SELECT * FROM scheme_performance", conn)
benchmarks = pd.read_sql("SELECT * FROM benchmark_indices", conn)
scheme_perf.head()


## 2. Return vs Risk Rating Analysis


In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=scheme_perf, x='return_3yr_pct', y='std_dev_ann_pct', hue='risk_grade', size='aum_crore', sizes=(50, 400), palette='Set1')
plt.title("3-Year Return vs Annualized Volatility")
plt.xlabel("3-Year Return (%)")
plt.ylabel("Annualized Standard Deviation (%)")
plt.show()


## 3. Sharpe & Sortino Ratio Comparisons


In [ ]:
top_sharpe = scheme_perf.sort_values('sharpe_ratio', ascending=False).head(10)
plt.figure(figsize=(10, 4))
sns.barplot(data=top_sharpe, x='sharpe_ratio', y='scheme_name', palette='viridis')
plt.title("Top 10 Schemes by Sharpe Ratio")
plt.xlabel("Sharpe Ratio")
plt.ylabel("Scheme Name")
plt.show()


## 4. NAV Trajectory vs Nifty 50 Benchmark


In [ ]:
nav_df = pd.read_sql("SELECT * FROM nav_history WHERE amfi_code = 125497", conn)
nav_df['date'] = pd.to_datetime(nav_df['date'], dayfirst=True)
nav_df = nav_df.sort_values('date')

nifty = benchmarks[benchmarks['index_name'] == 'NIFTY50'].copy()
nifty['date'] = pd.to_datetime(nifty['date'])
nifty = nifty.sort_values('date')

merged = pd.merge_asof(nav_df, nifty, on='date', direction='nearest')
merged['nav_norm'] = merged['nav'] / merged['nav'].iloc[0] * 100
merged['nifty_norm'] = merged['close_value'] / merged['close_value'].iloc[0] * 100

plt.figure(figsize=(12, 5))
plt.plot(merged['date'], merged['nav_norm'], label='Scheme 125497 (Indexed)', color='#4f6df5', linewidth=2)
plt.plot(merged['date'], merged['nifty_norm'], label='Nifty 50 (Indexed)', color='#ff9800', linestyle='--')
plt.title("Indexed Performance: Fund vs Benchmark")
plt.legend()
plt.show()
